# MCP: métodos estándar desde Jupyter

Este notebook está organizado alrededor de los **métodos estándar del protocolo MCP**.

La idea es ver la relación entre:

`Método MCP → Request/Result → función del SDK`

Servidor remoto de ejemplo:

`https://developers.openai.com/mcp`

## Flujo que vamos a seguir

`initialize → descubrimiento → lectura/prompts → tools/call → sampling/elicitation`

## 0. Mapa rápido

| Fase | Método MCP | Request / Result | SDK Python |
|---|---|---|---|
| Conexión | `initialize` | `InitializeRequest` → `InitializeResult` | `Client(...)` |
| Descubrimiento | `tools/list` | → `ListToolsResult` | `client.list_tools()` |
| Descubrimiento | `resources/list` | → `ListResourcesResult` | `client.list_resources()` |
| Descubrimiento | `resources/templates/list` | → `ListResourceTemplatesResult` | `client.list_resource_templates()` |
| Prompts | `prompts/list` | → `ListPromptsResult` | `client.list_prompts()` |
| Prompts | `prompts/get` | `GetPromptRequest` → `GetPromptResult` | `client.get_prompt()` |
| Datos | `resources/read` | `ReadResourceRequest` → `ReadResourceResult` | `client.read_resource()` |
| Ejecución | `tools/call` | `CallToolRequest` → `CallToolResult` | `client.call_tool()` |
| Sampling | `sampling/createMessage` | `CreateMessageRequest` → `CreateMessageResult` | callback cliente |
| Elicitation | `elicitation/create` | request → result | callback cliente |

> El SDK oculta el JSON-RPC y nos deja trabajar con funciones Python.

In [ ]:
%pip install -q "mcp>=2,<3"

In [ ]:
from __future__ import annotations

import json

from mcp import Client
from mcp.types import TextContent

OPENAI_DOCS_MCP = "https://developers.openai.com/mcp"

print("Servidor:", OPENAI_DOCS_MCP)

# 1. `initialize`

**Fase:** conexión.

Cuando abrimos una sesión MCP, cliente y servidor negocian versión, capacidades e información básica.

En el SDK no llamamos normalmente a `InitializeRequest` a mano: ocurre al entrar en `Client(...)`.

In [ ]:
async with Client(OPENAI_DOCS_MCP) as client:
    print("protocol_version:", client.protocol_version)
    print("server_info:", client.server_info)
    print("server_capabilities:", client.server_capabilities)

# 2. `tools/list`

**Fase:** descubrimiento.

Pregunta al servidor qué herramientas expone.

Resultado conceptual: `ListToolsResult`.

In [ ]:
async with Client(OPENAI_DOCS_MCP) as client:
    result = await client.list_tools()

    print("Tipo:", type(result).__name__)
    print("Número de tools:", len(result.tools))

    for tool in result.tools:
        print("\n---", tool.name, "---")
        print(tool.description)
        print(json.dumps(tool.input_schema, indent=2, ensure_ascii=False))

# 3. `resources/list`

**Fase:** descubrimiento.

Sirve para descubrir recursos concretos que el servidor pone a disposición del cliente.

Resultado conceptual: `ListResourcesResult`.

In [ ]:
async with Client(OPENAI_DOCS_MCP) as client:
    try:
        result = await client.list_resources()
        print("Tipo:", type(result).__name__)
        print(result)
    except Exception as exc:
        print("Este servidor no expone resources o no los soporta:")
        print(type(exc).__name__, exc)

# 4. `resources/templates/list`

**Fase:** descubrimiento.

Permite descubrir plantillas de recursos parametrizables.

Resultado conceptual: `ListResourceTemplatesResult`.

In [ ]:
async with Client(OPENAI_DOCS_MCP) as client:
    try:
        result = await client.list_resource_templates()
        print("Tipo:", type(result).__name__)
        print(result)
    except Exception as exc:
        print("No soportado por este servidor:")
        print(type(exc).__name__, exc)

# 5. `prompts/list`

**Fase:** descubrimiento de prompts.

Pregunta al servidor qué prompts reutilizables ofrece.

Resultado conceptual: `ListPromptsResult`.

In [ ]:
async with Client(OPENAI_DOCS_MCP) as client:
    try:
        result = await client.list_prompts()
        print("Tipo:", type(result).__name__)
        print(result)
    except Exception as exc:
        print("No soportado por este servidor:")
        print(type(exc).__name__, exc)

# 6. `prompts/get`

**Fase:** prompts.

Recupera un prompt concreto y, si aplica, le pasa argumentos.

Request / Result:

`GetPromptRequest → GetPromptResult`

La celda solo intenta ejecutarlo si el servidor expone prompts.

In [ ]:
async with Client(OPENAI_DOCS_MCP) as client:
    try:
        prompts = await client.list_prompts()

        if not prompts.prompts:
            print("El servidor no expone prompts.")
        else:
            prompt = prompts.prompts[0]
            print("Probando prompt:", prompt.name)

            result = await client.get_prompt(prompt.name)
            print("Tipo:", type(result).__name__)
            print(result)

    except Exception as exc:
        print("No disponible:")
        print(type(exc).__name__, exc)

# 7. `resources/read`

**Fase:** acceso a datos reales.

Lee el contenido de un recurso concreto.

Request / Result:

`ReadResourceRequest → ReadResourceResult`

Solo lo probamos si el servidor expone al menos un recurso.

In [ ]:
async with Client(OPENAI_DOCS_MCP) as client:
    try:
        resources = await client.list_resources()

        if not resources.resources:
            print("El servidor no expone resources.")
        else:
            resource = resources.resources[0]
            print("Leyendo:", resource.uri)

            result = await client.read_resource(resource.uri)
            print("Tipo:", type(result).__name__)
            print(result)

    except Exception as exc:
        print("No disponible:")
        print(type(exc).__name__, exc)

# 8. `tools/call`

**Fase:** ejecución.

Ejecuta una herramienta descubierta previamente.

Request / Result:

`CallToolRequest → CallToolResult`

In [ ]:
async with Client(OPENAI_DOCS_MCP) as client:
    tools = await client.list_tools()
    by_name = {tool.name: tool for tool in tools.tools}

    tool_name = "search_openai_docs"

    if tool_name not in by_name:
        print("Tool no encontrada. Disponibles:", list(by_name))
    else:
        result = await client.call_tool(
            tool_name,
            {"query": "Model Context Protocol tools"}
        )

        print("Tipo:", type(result).__name__)
        print("is_error:", result.is_error)

        for block in result.content:
            if isinstance(block, TextContent):
                print(block.text)
            else:
                print(block)

# 9. Sampling: `sampling/createMessage`

**Dirección diferente:** servidor → cliente.

Aquí el MCP Server pide al cliente que use un modelo y genere una respuesta.

Conceptualmente:

`CreateMessageRequest → CreateMessageResult`

Esto no suele aparecer como una llamada normal tipo `client.call_tool()`: el cliente debe registrar un **callback de sampling** para responder a la petición del servidor.

# 10. Elicitation: `elicitation/create`

También es una interacción iniciada por el servidor.

El servidor puede pedir información adicional al usuario a través del cliente.

Por tanto, igual que sampling, se implementa normalmente mediante **callbacks del cliente**, no con una llamada directa al servidor.

# 11. Resumen mental

## Cliente → Servidor

```text
initialize
tools/list
resources/list
resources/templates/list
prompts/list
prompts/get
resources/read
tools/call
```

## Servidor → Cliente

```text
sampling/createMessage
elicitation/create
```

La distinción importante es:

- **método MCP** = operación estándar del protocolo;
- **Request/Result** = tipos de mensajes;
- **SDK** = funciones Python cómodas que construyen esos mensajes por nosotros.

# 12. El transporte es otra capa

Los mismos métodos MCP pueden viajar por diferentes transportes.

```text
                    MÉTODOS MCP

       tools/list
       tools/call
       resources/read
       prompts/get
             │
      ┌──────┴────────┐
      │               │
    stdio      Streamable HTTP
    local           remoto
```

Primero entiende los **métodos MCP**. Después, el transporte.

# 13. Request vs Result: qué información aparece en cada lado

Una de las preguntas de la reunión era si la identificación del MCP puede mejorar entre el **request inicial** y el **result**.

Para verlo de forma sencilla, aquí separamos conceptualmente:

- **Request**: qué envía el cliente.
- **Result**: qué devuelve el servidor.

En `initialize`, el resultado suele aportar información especialmente útil para identificar al servidor:
- `serverInfo.name`
- `serverInfo.version`
- `capabilities`

In [ ]:
async with Client(OPENAI_DOCS_MCP) as client:
    print("=== RESULTADO DE INITIALIZE ===")
    print("protocol_version:", client.protocol_version)
    print("server_info:", client.server_info)
    print("server_capabilities:", client.server_capabilities)

    print("\nCampos útiles para identificación:")
    if client.server_info:
        print("- server name:", getattr(client.server_info, "name", None))
        print("- server version:", getattr(client.server_info, "version", None))

## Interpretación

Esto permite comprobar una hipótesis relevante:

```text
Request inicial
    ↓
puede contener URL / cliente / endpoint

Initialize Result
    ↓
puede añadir identidad del servidor + versión + capabilities
```

Por tanto, analizar solo un evento aislado puede dar una visión incompleta de la sesión.

# 14. Timeline de una sesión MCP

Para analizar correctamente una sesión, conviene pensarla como una secuencia de eventos y no como registros independientes.

El siguiente ejemplo genera un timeline simple de operaciones MCP.

In [ ]:
from datetime import datetime

timeline = []

def log_event(direction, method, detail=""):
    timeline.append({
        "timestamp": datetime.now().isoformat(timespec="milliseconds"),
        "direction": direction,
        "method": method,
        "detail": detail,
    })

async with Client(OPENAI_DOCS_MCP) as client:
    log_event("client → server", "initialize", "Conexión / negociación")
    log_event("server → client", "InitializeResult", str(client.server_info))

    tools = await client.list_tools()
    log_event("client → server", "tools/list", "Descubrimiento")
    log_event("server → client", "ListToolsResult", f"{len(tools.tools)} tools")

    tool_name = "search_openai_docs"
    if any(t.name == tool_name for t in tools.tools):
        log_event("client → server", "tools/call", tool_name)

        result = await client.call_tool(
            tool_name,
            {"query": "Model Context Protocol"}
        )

        log_event(
            "server → client",
            "CallToolResult",
            f"is_error={result.is_error}"
        )

for event in timeline:
    print(
        event["timestamp"],
        "|",
        event["direction"].ljust(15),
        "|",
        event["method"].ljust(18),
        "|",
        event["detail"]
    )

## Qué demuestra este timeline

Una única interacción puede generar varios eventos:

`initialize → result → tools/list → result → tools/call → result`

Esto ayuda a entender por qué una plataforma de seguridad puede necesitar **correlacionar eventos de la misma sesión** para construir una identificación final del MCP.

# 15. Discovery vs Execution

Para políticas de seguridad resulta útil separar operaciones de **descubrimiento** de operaciones que ya implican interacción efectiva con datos o sistemas.

Una clasificación simple sería:

In [ ]:
MCP_OPERATION_CLASSIFICATION = {
    "initialize": "Discovery",
    "tools/list": "Discovery",
    "resources/list": "Discovery",
    "resources/templates/list": "Discovery",
    "prompts/list": "Discovery",

    "prompts/get": "Effective interaction",
    "resources/read": "Effective interaction",
    "tools/call": "Execution",
}

for method, category in MCP_OPERATION_CLASSIFICATION.items():
    print(f"{method:28} -> {category}")

Una política podría, conceptualmente, tratar de forma distinta:

```text
Discovery
    initialize
    tools/list
    resources/list
    prompts/list

Effective interaction / execution
    prompts/get
    resources/read
    tools/call
```

Esto permite plantear un modelo donde un MCP todavía desconocido pueda ser descubierto, pero no ejecutar acciones hasta que exista más confianza sobre él.

# 16. Qué podemos validar con este notebook

| Prueba | Pregunta que ayuda a responder |
|---|---|
| Comparar `initialize` y `InitializeResult` | ¿Cuándo aparece información suficiente para identificar el MCP? |
| Ver `server_info` y capabilities | ¿Qué atributos aporta el servidor después de conectar? |
| Timeline de la sesión | ¿Es necesario correlacionar varios eventos? |
| `tools/list` | ¿Podemos permitir discovery sin ejecutar acciones? |
| `resources/read` | ¿Cuándo empieza acceso efectivo a datos? |
| `tools/call` | ¿Cuándo empieza una acción real sobre un sistema? |
| Clasificación Discovery vs Execution | ¿Tiene sentido aplicar políticas diferentes por tipo de operación? |

## Límite del notebook

Este notebook permite estudiar el **protocolo MCP** y sus eventos.

No demuestra por sí solo:
- cómo Netskope correlaciona esos eventos;
- cómo calcula CCL;
- cómo clasifica MCPs Unknown;
- qué políticas concretas puede aplicar;
- ni qué licencias son necesarias.

Esas cuestiones requieren validación específica en Netskope.